In [4]:
import pandas as pd
import numpy as np
import sqlite3
import json

# Reproducibility ke liye seed set karo
np.random.seed(42)
n_samples = 500

# Base DataFrame generate karo
data = {
    'customer_id': [f"CUST_{1000+i}" for i in range(n_samples)],
    'age': np.random.choice([np.nan, 22, 28, 35, 45, 50, 62], size=n_samples, p=[0.1, 0.15, 0.2, 0.25, 0.15, 0.1, 0.05]),
    'gender': np.random.choice(['Male', 'Female', 'Other', None], size=n_samples, p=[0.45, 0.45, 0.05, 0.05]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], size=n_samples),
    'education_level': np.random.choice(['Primary', 'Secondary', 'Graduate', 'Post-Graduate'], size=n_samples),
    'employment_type': np.random.choice(['Salaried', 'Self-Employed', 'Unemployed', None], size=n_samples, p=[0.5, 0.35, 0.1, 0.05]),
    'annual_income': np.random.choice([np.nan, 300000, 500000, 800000, 1200000, 2500000, 10000000], size=n_samples, p=[0.1, 0.2, 0.3, 0.2, 0.1, 0.07, 0.03]),
    'loan_amount': np.random.exponential(scale=200000, size=n_samples) + 50000,
    'loan_purpose': np.random.choice(['Home', 'Car', 'Education', 'Business', 'Other'], size=n_samples),
    'credit_score': np.random.choice([np.nan, 350, 580, 650, 720, 810], size=n_samples, p=[0.1, 0.1, 0.2, 0.3, 0.2, 0.1]),
    'repayment_history': np.random.randint(0, 6, size=n_samples),
    'transaction_count': np.random.randint(5, 150, size=n_samples),
    'spending_ratio': np.random.uniform(5, 95, size=n_samples),
    'join_date': pd.date_range(start='2020-01-01', periods=n_samples, freq='D').strftime('%Y-%m-%d'),
    'default_flag': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
}

df_full = pd.DataFrame(data)

# ----------------------------------------------------
# Multi-source simulation ke liye dataset split & save
# ----------------------------------------------------

# 1. Main transactions & numericals -> CSV
df_full[['customer_id', 'age', 'annual_income', 'loan_amount', 'credit_score', 'spending_ratio']].to_csv('Dataset/main_transactions.csv', index=False)

# 2. Customer metadata & demographics -> JSON
metadata = df_full[['customer_id', 'gender', 'region', 'education_level', 'employment_type', 'join_date']].to_dict(orient='records')
with open('Dataset/customer_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

# 3. Loan & Repayment history -> SQLite Database
conn = sqlite3.connect('Dataset/loan_data.db')
df_full[['customer_id', 'loan_purpose', 'repayment_history', 'transaction_count', 'default_flag']].to_sql('repayment_history', conn, if_exists='replace', index=False)
conn.close()

print("Files generated successfully: CSV, JSON, and SQLite Database!")

Files generated successfully: CSV, JSON, and SQLite Database!
